## 01 - Data Understanding: Online Retail

This notebook is a first look at the Online Retail dataset before any cleaning. I want to know what each column means, how complete the data is, and which rows are not regular sales, so the cleaning step later is based on evidence instead of guesses.

The data comes from a UK-based online store that sells gift items, mostly to wholesalers, and covers 1 Dec 2010 to 9 Dec 2011. Source: [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/352/online+retail).

Questions to answer here:
- What does each column mean, and are the data types right?
- How many values are missing or duplicated?
- Which rows are not regular sales (cancellations, stock adjustments, zero or negative prices, non-product codes)?
- Are there extreme values that could distort the analysis?

## 1. Load the data

The raw file is not tracked in git. Download it from UCI and save it as `data/raw/Online_Retail.xlsx`.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

DATA_PATH = "../data/raw/Online Retail.xlsx"
retail_data = pd.read_excel(DATA_PATH)

retail_data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 2. Data dictionary

| Column | Meaning | Type when loaded |
|---|---|---|
| `InvoiceNo` | Invoice number. A leading `C` means the invoice is a cancellation | object |
| `StockCode` | Product code | object |
| `Description` | Product name | object |
| `Quantity` | Number of items on that line | int |
| `InvoiceDate` | Date and time of the transaction | datetime |
| `UnitPrice` | Price per item (GBP) | float |
| `CustomerID` | Customer ID | float |
| `Country` | Customer's country | str |

One row is one product line on an invoice, not a whole invoice. `CustomerID` is loaded as float only because it has missing values, so I'll convert it to a string or `Int64` later.

## 3. Basic overview

In [2]:
retail_data.shape

(541909, 8)

In [3]:
retail_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [4]:
retail_data[["Quantity", "UnitPrice"]].describe(percentiles=[.01, .25, .5, .75, .99])

,Quantity,UnitPrice
count,541909.000000,541909.000000
mean,9.552250,4.611114
std,218.081158,96.759853
min,-80995.000000,-11062.060000
1%,-2.000000,0.190000
25%,1.000000,1.250000
50%,3.000000,2.080000
75%,10.000000,4.130000
99%,100.000000,18.000000
max,80995.000000,38970.000000


In [5]:
print("Date range:", retail_data["InvoiceDate"].min(), "to", retail_data["InvoiceDate"].max())
retail_data[["InvoiceNo", "StockCode", "CustomerID", "Country"]].nunique()

Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


InvoiceNo     25900
StockCode      4070
CustomerID     4372
Country          38
dtype: int64

There are 541,909 rows and 8 columns, with 25,900 unique invoices (cancellations included), 4,070 stock codes, 4,372 customers and 38 countries.

Two things already stand out. `Quantity` goes from -80,995 to 80,995 and `UnitPrice` from -11,062 to 38,970, so both columns have negative and extreme values and can't be used for revenue as they are. Also, the data stops on 9 Dec 2011, which means December 2011 is only a partial month.

## 4. Missing values

In [6]:
retail_data.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [7]:
missing_percentage = retail_data.isnull().mean() * 100
print(missing_percentage)

InvoiceNo       0.000000
StockCode       0.000000
Description     0.268311
Quantity        0.000000
InvoiceDate     0.000000
UnitPrice       0.000000
CustomerID     24.926694
Country         0.000000
dtype: float64


`CustomerID` is missing in 135,080 rows (24.93%) and `Description` in 1,454 rows (0.27%). The other columns are complete.

Before moving on I want to know if the missing `CustomerID` is random or tied to something, so let's break it down by country.

In [8]:
(
    retail_data.assign(missing_customer=retail_data["CustomerID"].isnull())
    .groupby("Country")["missing_customer"]
    .mean()
    .sort_values(ascending=False)
    .head(5)
)

Country
Hong Kong         1.000000
Unspecified       0.452915
United Kingdom    0.269639
Israel            0.158249
Bahrain           0.105263
Name: missing_customer, dtype: float64

It's not random. Hong Kong has no `CustomerID` at all, Unspecified is missing 45% and the United Kingdom 27%. For anything customer-based (RFM for example) roughly a quarter of the data can't be used, but those rows are still fine for sales or product analysis.

## 5. Duplicates

In [9]:
duplicate_rows = retail_data[retail_data.duplicated()]
print("Duplicate rows (excluding the first occurrence):", duplicate_rows.shape[0])

Duplicate rows (excluding the first occurrence): 5268


In [10]:
all_duplicate_rows = retail_data[retail_data.duplicated(keep=False)]
print("All rows involved in duplicates:", all_duplicate_rows.shape[0])
print("Of those, rows with a CustomerID:", all_duplicate_rows["CustomerID"].notna().sum())

All rows involved in duplicates: 10147
Of those, rows with a CustomerID: 10062


There are 5,268 exact duplicate rows, or 10,147 if the first occurrence of each is counted too. Almost all of them (10,062) have a `CustomerID`, so unlike the stock adjustments in the next section these look like real customer lines, probably double entries or the same item scanned twice. The data alone can't confirm that, so I'll decide in the cleaning notebook and write down the reason there.

6. Negative quantities

A negative `Quantity` should mean a cancellation, and cancellation invoices start with the letter C. Let's check whether that actually holds.

In [11]:
negative_quantity_data = retail_data[retail_data["Quantity"] < 0]
print("Total negative Quantity rows:", negative_quantity_data.shape[0])

Total negative Quantity rows: 10624


In [12]:
invoice_is_cancellation = (
    retail_data["InvoiceNo"]
    .astype("string")
    .str.startswith("C", na=False)
)

negative_cancellation_data = retail_data[
    (retail_data["Quantity"] < 0) & invoice_is_cancellation
]
negative_non_cancellation_data = retail_data[
    (retail_data["Quantity"] < 0) & (~invoice_is_cancellation)
]

print("Negative and a C invoice (cancellations):", negative_cancellation_data.shape[0])
print("Negative but NOT a C invoice:", negative_non_cancellation_data.shape[0])

Negative and a C invoice (cancellations): 9288
Negative but NOT a C invoice: 1336


The C rule explains most of it, but 1,336 negative rows are not cancellations. Let's look at those.

In [13]:
negative_non_cancellation_data[
    ["InvoiceNo", "Quantity", "InvoiceDate", "CustomerID"]
].head(10)

,InvoiceNo,Quantity,InvoiceDate,CustomerID
2406,536589,-10,2010-12-01 16:50:00,NaN
4347,536764,-38,2010-12-02 14:42:00,NaN
7188,536996,-20,2010-12-03 15:30:00,NaN
7189,536997,-20,2010-12-03 15:30:00,NaN
7190,536998,-6,2010-12-03 15:30:00,NaN
7192,537000,-22,2010-12-03 15:32:00,NaN
7193,537001,-6,2010-12-03 15:33:00,NaN
7195,537003,-2,2010-12-03 15:33:00,NaN
7196,537004,-30,2010-12-03 15:34:00,NaN
7197,537005,-70,2010-12-03 15:35:00,NaN


In [14]:
print("Missing CustomerID:", negative_non_cancellation_data["CustomerID"].isnull().sum())
print("Unique invoices:", negative_non_cancellation_data["InvoiceNo"].nunique())
print("UnitPrice = 0:", (negative_non_cancellation_data["UnitPrice"] == 0).sum())
print("Missing Description:", negative_non_cancellation_data["Description"].isnull().sum())

Missing CustomerID: 1336
Unique invoices: 1336
UnitPrice = 0: 1336
Missing Description: 862


In [15]:
negative_non_cancellation_data[negative_non_cancellation_data["Description"].notna()].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
7313,537032,21275,?,-30,2010-12-03 16:50:00,0.0,NaN,United Kingdom
13217,537425,84968F,check,-20,2010-12-06 15:35:00,0.0,NaN,United Kingdom
13218,537426,84968E,check,-35,2010-12-06 15:36:00,0.0,NaN,United Kingdom
13264,537432,35833G,damages,-43,2010-12-06 16:10:00,0.0,NaN,United Kingdom
21338,538072,22423,faulty,-13,2010-12-09 14:10:00,0.0,NaN,United Kingdom
21518,538090,20956,?,-723,2010-12-09 14:48:00,0.0,NaN,United Kingdom
22296,538161,46000S,Dotcom sales,-100,2010-12-09 17:25:00,0.0,NaN,United Kingdom
22297,538162,46000M,Dotcom sales,-100,2010-12-09 17:25:00,0.0,NaN,United Kingdom
42564,540010,22501,reverse 21/5/10 adjustment,-100,2011-01-04 11:13:00,0.0,NaN,United Kingdom
42566,540012,22502,reverse 21/5/10 adjustment,-100,2011-01-04 11:14:00,0.0,NaN,United Kingdom


The descriptions here look like notes typed by hand (check, damaged, thrown away, ?). I'll lowercase them so "Damaged" and "damaged" are counted together.

In [16]:
normalized_description_counts = (
    negative_non_cancellation_data["Description"].str.lower().value_counts().head(10)
)
print(normalized_description_counts)
print("Unique descriptions (lowercase):",
      negative_non_cancellation_data["Description"].str.lower().nunique())

Description
check                     123
damaged                    57
damages                    46
?                          41
sold as set on dotcom      20
thrown away                 9
unsaleable, destroyed.      9
??                          7
damages?                    5
wet damaged                 5
Name: count, dtype: int64
Unique descriptions (lowercase): 126


Regular cancellations can also lack a `CustomerID`, so let's check that too.

In [17]:
print("Cancellations without CustomerID:",
      (invoice_is_cancellation & retail_data["CustomerID"].isnull()).sum())

Cancellations without CustomerID: 383


And the most extreme quantities, positive or negative (70,000 units or more):

In [18]:
retail_data[retail_data["Quantity"].abs() >= 70000]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom
61624,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346.0,United Kingdom
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom


Of the 10,624 negative rows, 9,288 are regular cancellations and 1,336 are something else. Those 1,336 are very uniform: `UnitPrice` is 0, there is no `CustomerID`, every row is its own invoice, and the descriptions are hand-written notes. That looks like an internal stock adjustment (damaged or lost goods) rather than a sale, so I'll keep it out of any sales analysis.

Two other things. 383 cancellations have no `CustomerID`, so not every cancellation can be traced back to a customer. And there are two order-then-cancel pairs with huge quantities: 541431 (+74,215) cancelled by C541433 for customer 12346, and 581483 (+80,995) cancelled 12 minutes later by C581484 for customer 16446. Each pair cancels itself out, but the second one is what stretches the revenue range to ±168,469.6 (80,995 x 2.08).

## 7. Invoice prefixes and negative prices

Invoice numbers should be plain digits, optionally with a C in front. Let's see if there are other prefixes.

In [19]:
first_char = retail_data["InvoiceNo"].astype("string").str[0]
print(first_char.where(~first_char.str.isdigit(), "digit").value_counts())

InvoiceNo
digit    532618
C          9288
A             3
Name: count, dtype: Int64


In [20]:
retail_data[retail_data["InvoiceNo"].astype("string").str.startswith("A", na=False)]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299982,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [21]:
retail_data[retail_data["UnitPrice"] < 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


There is a third kind of invoice, the ones starting with A. All three are "Adjust bad debt" lines with no `CustomerID`: one is +11,062.06 (A563185) and the other two are -11,062.06 (A563186 and A563187). Those two are the only negative `UnitPrice` values in the whole dataset. They are accounting entries, not product sales, so they have to be excluded from revenue.

## 8. Zero unit price

The 1,336 stock adjustments from section 6 all have a price of 0, so let's look at zero prices in general.

In [22]:
zero_unit_price_count = (retail_data["UnitPrice"] == 0).sum()
print("Total rows with UnitPrice = 0:", zero_unit_price_count)

Total rows with UnitPrice = 0: 2515


In [23]:
zero_unit_price_data = retail_data[
    (retail_data["UnitPrice"] == 0) & (retail_data["Quantity"] > 0)
]
print("UnitPrice = 0 with positive Quantity:", zero_unit_price_data.shape[0])

UnitPrice = 0 with positive Quantity: 1179


In [24]:
zero_unit_price_data[["Description", "StockCode", "InvoiceNo", "CustomerID"]].head(10)

,Description,StockCode,InvoiceNo,CustomerID
622,NaN,22139,536414,NaN
1970,NaN,21134,536545,NaN
1971,NaN,22145,536546,NaN
1972,NaN,37509,536547,NaN
1987,NaN,85226A,536549,NaN
1988,NaN,85044,536550,NaN
2024,NaN,20950,536552,NaN
2025,NaN,37461,536553,NaN
2026,NaN,84670,536554,NaN
4348,NaN,84952C,536765,NaN


In [25]:
zero_unit_price_data[["Description", "CustomerID"]].isnull().sum()

Description     592
CustomerID     1139
dtype: int64

A few of them do have a `CustomerID`, so let's see who they are.

In [26]:
identified_customers_data = zero_unit_price_data[zero_unit_price_data["CustomerID"].notna()]
print("Rows:", identified_customers_data.shape[0])
print("Unique customers:", identified_customers_data["CustomerID"].nunique())
print("Unique invoices:", identified_customers_data["InvoiceNo"].nunique())

Rows: 40
Unique customers: 31
Unique invoices: 34


In [27]:
invoice_line_count = identified_customers_data.groupby("InvoiceNo").size()
multi_line_invoices = invoice_line_count[invoice_line_count > 1]
print(multi_line_invoices)
print("Total invoices:", multi_line_invoices.shape[0])

InvoiceNo
540372    2
564651    4
574920    2
575579    2
dtype: int64
Total invoices: 4


In [28]:
multi_line_invoice_details = identified_customers_data[
    identified_customers_data["InvoiceNo"].isin(multi_line_invoices.index)
][["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "CustomerID"]].sort_values("InvoiceNo")

print(multi_line_invoice_details)

       InvoiceNo StockCode                          Description  Quantity  UnitPrice  CustomerID
47068     540372     22090              PAPER BUNTING RETROSPOT        24        0.0     13081.0
47070     540372     22553               PLASTERS IN TIN SKULLS        24        0.0     13081.0
314745    564651     23270     SET OF 2 CERAMIC PAINTED HEARTS         96        0.0     14646.0
314746    564651     23268  SET OF 2 CERAMIC CHRISTMAS REINDEER       192        0.0     14646.0
314747    564651     22955             36 FOIL STAR CAKE CASES        144        0.0     14646.0
314748    564651     21786                   POLKADOT RAIN HAT        144        0.0     14646.0
446793    574920     22899         CHILDREN'S APRON DOLLY GIRL          1        0.0     13985.0
446794    574920     23480       MINI LIGHTS WOODLAND MUSHROOMS         1        0.0     13985.0
454463    575579     22437        SET OF 9 BLACK SKULL BALLOONS        20        0.0     13081.0
454464    575579     22089    

In [29]:
quantity_by_invoice = identified_customers_data.groupby("InvoiceNo")["Quantity"].sum()
print(quantity_by_invoice.sort_values(ascending=False).head(5))

InvoiceNo
578841    12540
564651      576
562973      240
574138      216
554037       80
Name: Quantity, dtype: int64


In total 2,515 rows have `UnitPrice` = 0. 1,336 of them are the stock adjustments from section 6 and the other 1,179 have a positive `Quantity`. Of those 1,179, 1,139 have no `CustomerID` and 592 have no `Description`. Only 40 rows (31 customers, 34 invoices) can be identified, and they look like free items or samples.

One invoice stands out: 578841 with 12,540 units. I'll look into it in section 11.

## 9. Non-product stock codes

Normal product codes start with five digits (for example `85123A`). Let's see what doesn't follow that pattern.

In [30]:
stock_str = retail_data["StockCode"].astype("string")
non_product = retail_data[~stock_str.str.match(r"^\d{5}", na=False)]

print("Rows with a non-standard StockCode:", non_product.shape[0])
non_product["StockCode"].value_counts().head(10)

Rows with a non-standard StockCode: 2995


StockCode
POST            1256
DOT              710
M                571
C2               144
D                 77
S                 63
BANK CHARGES      37
AMAZONFEE         34
CRUK              16
DCGSSGIRL         13
Name: count, dtype: int64

In [31]:
non_product.groupby("StockCode")["Description"].first().loc[
    ["POST", "DOT", "M", "BANK CHARGES", "AMAZONFEE", "CRUK"]
]

StockCode
POST                    POSTAGE
DOT              DOTCOM POSTAGE
M                        Manual
BANK CHARGES       Bank Charges
AMAZONFEE            AMAZON FEE
CRUK            CRUK Commission
Name: Description, dtype: object

About 2,995 rows are not physical products. The biggest ones are `POST` (postage, 1,256 rows), `DOT` (dotcom postage, 710) and `M` (manual, 571), followed by `BANK CHARGES`, `AMAZONFEE` and `CRUK`. Left in, they would pollute any product analysis such as best sellers, so they need to be separated later.

## 10. Description consistency

Can `Description` be used to identify a product? Only if each `StockCode` has a single description.

In [32]:
desc_per_code = (
    retail_data.dropna(subset=["Description"])
    .groupby("StockCode")["Description"]
    .nunique()
)
print("StockCodes with more than one description:", (desc_per_code > 1).sum(), "out of", desc_per_code.shape[0])

StockCodes with more than one description: 650 out of 3958


It can't. 650 of 3,958 stock codes have more than one description, so `StockCode` is the key to use, and I'll pick one standard description per code during cleaning.

## 11. Case study: StockCode 84826

Invoice 578841 showed up in section 8 with 12,540 units at a price of 0. Is that normal for this product?

In [33]:
retail_data[retail_data["InvoiceNo"] == 578841][
    ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "CustomerID"]
]

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID
502122,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,0.0,13256.0


In [34]:
product_84826_data = retail_data[retail_data["StockCode"] == 84826].sort_values(
    "Quantity", ascending=False
)
print(product_84826_data.head(10))
product_84826_data["Quantity"].describe()

       InvoiceNo StockCode                     Description  Quantity         InvoiceDate  UnitPrice  CustomerID         Country
502122    578841     84826  ASSTD DESIGN 3D PAPER STICKERS     12540 2011-11-25 15:57:00       0.00     13256.0  United Kingdom
47846     540420     84826  ASSTD DESIGN 3D PAPER STICKERS       120 2011-01-07 11:13:00       0.21     12829.0  United Kingdom
31105     538895     84826  ASSTD DESIGN 3D PAPER STICKERS        60 2010-12-15 09:07:00       0.21     14401.0  United Kingdom
30536     538870     84826  ASSTD DESIGN 3D PAPER STICKERS        60 2010-12-14 14:54:00       0.21     12829.0  United Kingdom
3323      536620     84826  ASSTD DESIGN 3D PAPER STICKERS        60 2010-12-02 10:27:00       0.85     14135.0  United Kingdom
29521     538801     84826  ASSTD DESIGN 3D PAPER STICKERS        60 2010-12-14 11:51:00       0.21     16083.0  United Kingdom
89638     543915     84826  ASSTD DESIGN 3D PAPER STICKERS        60 2011-02-14 13:21:00       0.21     

count       38.000000
mean       359.078947
std       2029.704457
min        -60.000000
25%          1.250000
50%         22.000000
75%         60.000000
max      12540.000000
Name: Quantity, dtype: float64

IQR check on this product only:

In [35]:
q1_84826 = product_84826_data["Quantity"].quantile(0.25)
q2_84826 = product_84826_data["Quantity"].quantile(0.5)
q3_84826 = product_84826_data["Quantity"].quantile(0.75)
iqr_84826 = q3_84826 - q1_84826
upper_bound_84826 = q3_84826 + 1.5 * iqr_84826
lower_bound_84826 = q1_84826 - 1.5 * iqr_84826

print("Q1:", q1_84826)
print("Q2:", q2_84826)
print("Q3:", q3_84826)
print("IQR:", iqr_84826)
print("Upper Bound:", upper_bound_84826)
print("Lower Bound:", lower_bound_84826)

Q1: 1.25
Q2: 22.0
Q3: 60.0
IQR: 58.75
Upper Bound: 148.125
Lower Bound: -86.875


In [36]:
outlier_84826 = product_84826_data[
    (product_84826_data["Quantity"] > upper_bound_84826)
    | (product_84826_data["Quantity"] < lower_bound_84826)
]
outlier_84826[["InvoiceNo", "StockCode", "Quantity", "UnitPrice", "CustomerID"]]

,InvoiceNo,StockCode,Quantity,UnitPrice,CustomerID
502122,578841,84826,12540,0.0,13256.0


Does customer 13256 have any other purchases?

In [37]:
customer_13256_data = retail_data[retail_data["CustomerID"] == 13256].sort_values(
    "InvoiceDate", ascending=False
)
print(customer_13256_data)

       InvoiceNo StockCode                     Description  Quantity         InvoiceDate  UnitPrice  CustomerID         Country
502122    578841     84826  ASSTD DESIGN 3D PAPER STICKERS     12540 2011-11-25 15:57:00        0.0     13256.0  United Kingdom


What prices does this product usually sell at, and how much of the volume came from the zero-price row?

In [38]:
price_frequency = product_84826_data["UnitPrice"].value_counts()
print(price_frequency)

zero_price_percentage = (product_84826_data["UnitPrice"] == 0).mean() * 100
print("Share of rows with price 0 (%):", zero_price_percentage)

UnitPrice
0.85    22
0.21    13
0.19     2
0.00     1
Name: count, dtype: int64
Share of rows with price 0 (%): 2.631578947368421


In [39]:
quantity_by_price = product_84826_data.groupby("UnitPrice")["Quantity"].sum()
print(quantity_by_price)

zero_price_quantity_percentage = quantity_by_price.loc[0.0] / quantity_by_price.sum() * 100
print("Share of quantity from price 0 (%):", zero_price_quantity_percentage)

UnitPrice
0.00    12540
0.19       92
0.21      840
0.85      173
Name: Quantity, dtype: int64
Share of quantity from price 0 (%): 91.90179552949799


Effect on the mean and median once the row above the IQR upper bound is removed:

In [40]:
without_outlier = product_84826_data[product_84826_data["Quantity"] <= upper_bound_84826]["Quantity"]

print("Mean   before:", product_84826_data["Quantity"].mean(), "| after:", without_outlier.mean())
print("Median before:", product_84826_data["Quantity"].median(), "| after:", without_outlier.median())

Mean   before: 359.07894736842104 | after: 29.864864864864863
Median before: 22.0 | after: 12.0


Is this a one-off, or does it happen across the whole dataset? Here I only look at sales-like rows (`Quantity` > 0 and `UnitPrice` > 0).

In [41]:
sales_like = retail_data[(retail_data["Quantity"] > 0) & (retail_data["UnitPrice"] > 0)]

q1, q3 = sales_like["Quantity"].quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)

print("IQR upper bound (Quantity):", upper)
print("Rows above the bound:", (sales_like["Quantity"] > upper).sum(),
      "out of", sales_like.shape[0])
print("99.9th percentile:", sales_like["Quantity"].quantile(0.999))
sales_like["Quantity"].nlargest(5)

IQR upper bound (Quantity): 23.5
Rows above the bound: 56363 out of 530104
99.9th percentile: 449.7939999999944


540421    80995
61619     74215
421632     4800
206121     4300
97432      3906
Name: Quantity, dtype: int64

Invoice 578841 (customer 13256, price 0) is the only outlier for this product according to IQR. It accounts for 91.9% of the product's total quantity but only 2.6% of its rows, and it pushes the mean from 29.9 up to 359.1. The median moves from 22 to 12, much less than the mean, so it's the better summary for this kind of data.

Across the whole dataset the IQR upper bound is only 23.5 units, and it flags 56,363 of 530,104 rows (10.6%). Most of those are probably legit wholesale orders, since `Quantity` is very right-skewed (the 99.9th percentile is around 450). So IQR alone is not a good rule for deleting rows. I'd rather flag outliers and decide case by case. The top two values (80,995 and 74,215) are the cancelled pairs from section 6, so they aren't real sales anyway.

## 12. Country and time

A quick look at where the orders come from and how they are spread over time.

In [42]:
retail_data["Country"].value_counts(normalize=True).head(5)

Country
United Kingdom    0.914320
Germany           0.017521
France            0.015790
EIRE              0.015124
Spain             0.004674
Name: proportion, dtype: float64

In [43]:
monthly_invoices = retail_data.set_index("InvoiceDate").resample("ME")["InvoiceNo"].nunique()
monthly_invoices

InvoiceDate
2010-12-31    2025
2011-01-31    1476
2011-02-28    1393
2011-03-31    1983
2011-04-30    1744
2011-05-31    2162
2011-06-30    2012
2011-07-31    1927
2011-08-31    1737
2011-09-30    2327
2011-10-31    2637
2011-11-30    3462
2011-12-31    1015
Freq: ME, Name: InvoiceNo, dtype: int64

The United Kingdom makes up about 91.4% of the rows, so any country comparison will rest on small samples for most of the other countries. Invoices per month climb to a peak in November 2011 (3,462), which fits the pre-Christmas season. December 2011 only has 1,015 invoices, but that's because the data ends on 9 Dec and not because sales dropped, so I'll leave that month out of any monthly trend.

## 13. Summary and cleaning plan

These are the issues found and what I plan to do with each one in the cleaning notebook. They're first decisions and might change once I start cleaning.

| # | Issue | Rows | Plan |
|---|---|---|---|
| 1 | Missing `CustomerID` | 135,080 | Keep for sales analysis, drop for customer analysis |
| 2 | Missing `Description` | 1,454 | Fill from `StockCode` where possible, drop the rest |
| 3 | Exact duplicate rows | 5,268 | Decide and document the reason |
| 4 | Cancellations (`C` invoices) | 9,288 | Move to a separate table |
| 5 | Stock adjustments (negative `Quantity`, not `C`, price 0) | 1,336 | Exclude from sales analysis |
| 6 | `A` invoices, "Adjust bad debt" (two with negative price) | 3 | Exclude |
| 7 | Non-product `StockCode` (`POST`, `DOT`, `M`, etc.) | about 2,995 | Separate from product analysis |
| 8 | Zero price with positive `Quantity` | 1,179 | Flag and exclude from revenue |
| 9 | Extreme values: cancelled pairs of ±80,995 and ±74,215, plus 12,540 units at price 0 | 5 | Flag and exclude |
| 10 | Several `Description` values per `StockCode` | 650 codes | Pick one standard description per code |
| 11 | December 2011 is incomplete | n/a | Exclude from monthly trends |
| 12 | `CustomerID` stored as float | n/a | Convert to string or `Int64` |

Next up is `02_data_cleaning.ipynb`, which applies these decisions and saves the result to `data/processed/`.